<a href="https://colab.research.google.com/github/nithin12342/phase2/blob/main/ml_pipeline/h5_omnifusion/notebooks/H5_OmniFusion_Colab_Runner_YouTube.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧠 H5-OmniFusion Pipeline - YouTube Runner

**Process YouTube Videos from CSV**

Input: `dvlog-video-links.csv`
Output: `H5` files in Google Drive

---

## Quick Start
1. Run all cells
2. Pipeline loads CSV, downloads YouTube videos, and processes them

In [ ]:
# 1. Clone fresh
!git clone https://github.com/nithin12342/phase2.git /content/phase2

# 2. Install dependencies (including yt-dlp)
!pip install -q yt-dlp

# 3. Check Commit
!cd /content/phase2 && git rev-parse --short HEAD

In [ ]:
# @title ⏰ Anti-Disconnect Keep Alive
# Run this cell to prevent Colab from disconnecting due to inactivity.
import IPython
from google.colab import output

display(IPython.display.Javascript('''
  function ClickConnect(){
    console.log("KeepAlive: Working..."); 
    document.querySelector("colab-connect-button").shadowRoot.querySelector("#connect").click(); 
  }
  setInterval(ClickConnect, 60000);
'''))

print("\u2705 Keep Alive Activated! Run this cell and keep the tab open.")

In [ ]:
import os
# 1. Clone if it doesn't exist, or Pull if it does
if not os.path.exists("phase2"):
    !git clone https://github.com/nithin12342/phase2.git
    %cd phase2
else:
    %cd phase2
    !git pull origin main

# 2. Verify we are in the right place
!ls -la

## Cell 1: Install Dependencies

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")

!pip install -q transformers>=4.36.0 timm einops huggingface_hub --quiet
!pip install -q librosa soundfile praat-parselmouth noisereduce --quiet
!pip install -q mediapipe nltk vaderSentiment snownlp h5py tqdm --quiet
!pip install -q opensmile --quiet 2>/dev/null || echo "OpenSMILE fallback"
!pip install -q pandas tabulate --quiet

import nltk
try:
    nltk.download('vader_lexicon')
    nltk.download('punkt')
except:
    pass

print("\n✅ Dependencies installed. RESTART RUNTIME if needed.")

## Cell 2: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Cell 4: Add Paths

In [ ]:
import sys
PREPROCESSING_PATH = '/content/phase2/ml_pipeline/h5_omnifusion/preprocessing_and_feature_extraction'
SRC_PATH = '/content/phase2/ml_pipeline/h5_omnifusion/src'
for p in [PREPROCESSING_PATH, SRC_PATH]:
    if p not in sys.path: sys.path.insert(0, p)
print('Paths added')

## Cell 5: Configuration

In [ ]:
from dataclasses import dataclass
from typing import Tuple
import os, torch

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

@dataclass
class Config:
    BASE_PATH: str = '/content/drive/MyDrive/DAIC-WOZ_Datasets'
    PRETRAINED_PATH: str = '/content/drive/MyDrive/DAIC-WOZ_Datasets/pretrained_models'
    OUTPUT_PATH: str = '/content/drive/MyDrive/DAIC-WOZ_Datasets/H5_OmniFusion_Output/YouTube'
    TEMP_PATH: str = '/content/temp_extraction'
    SAMPLE_RATE: int = 16000
    EMBED_DIM: int = 768
    # Missing paths required by H5OmniFusionPipeline init
    DAIC_WOZ_PATH: str = '/content/drive/MyDrive/DAIC-WOZ_Datasets/DAIC-WOZ'
    EXTENDED_DAIC_WOZ_PATH: str = '/content/drive/MyDrive/DAIC-WOZ_Datasets/Extended-DAIC-WOZ/data'
    EATD_CORPUS_PATH: str = '/content/drive/MyDrive/DAIC-WOZ_Datasets/EATD-Corpus/EATD-Corpus'
    OUTPUT_DAIC_WOZ: str = '/content/drive/MyDrive/DAIC-WOZ_Datasets/H5_OmniFusion_Output/DAIC-WOZ'
    OUTPUT_EXTENDED_DAIC: str = '/content/drive/MyDrive/DAIC-WOZ_Datasets/H5_OmniFusion_Output/Extended-DAIC'
    OUTPUT_EATD: str = '/content/drive/MyDrive/DAIC-WOZ_Datasets/H5_OmniFusion_Output/EATD-Corpus'
    DEVICE: str = DEVICE
    
CFG = Config()
os.makedirs(CFG.TEMP_PATH, exist_ok=True)
os.makedirs(CFG.OUTPUT_PATH, exist_ok=True)

print(f'✅ Config loaded. Device: {CFG.DEVICE}')
print(f'📂 Output: {CFG.OUTPUT_PATH}')

## Cell 6: Load Video List from CSV

In [ ]:
import pandas as pd
CSV_PATH = '/content/phase2/dvlog-video-links.csv'

if os.path.exists(CSV_PATH):
    df_videos = pd.read_csv(CSV_PATH)
    print(f'✅ Loaded {len(df_videos)} videos from CSV')
    print(df_videos.head())
else:
    print(f'❌ CSV not found at {CSV_PATH}')
    df_videos = pd.DataFrame()

## Cell 7: Import Pipeline

In [ ]:
from model_loader import ModelLoader
from pipeline_audio import AudioPreprocessor
from pipeline_text import TextPreprocessor
from pipeline_video_face import VideoPreprocessor, FacePreprocessor
from pipeline_fusion_main import H5OmniFusionPipeline
print('✅ Pipeline imported')

## Cell 8: Load Models

In [ ]:
loader = ModelLoader(CFG.DEVICE, pretrained_path=CFG.PRETRAINED_PATH)
loader.load_wav2vec2()
loader.load_text_encoder('english')
loader.load_videomae()
loader.load_face_encoder()
print(f'✅ Models: {list(loader.get_loaded_models().keys())}')

In [ ]:
from tqdm.auto import tqdm
audio_proc = AudioPreprocessor(loader)
text_proc = TextPreprocessor(loader, CFG.EMBED_DIM, str(CFG.DEVICE))
video_proc = VideoPreprocessor(loader, CFG.EMBED_DIM)
face_proc = FacePreprocessor(loader, CFG.EMBED_DIM)

pipeline = H5OmniFusionPipeline(audio_proc, text_proc, video_proc, face_proc, CFG)
print('✅ Preprocessors ready')

## Cell 9: YouTube Processing Function

In [ ]:
import glob
import subprocess
import shutil
import numpy as np
import os
import time
from tqdm.auto import tqdm

def process_youtube_participant(pipeline, pid, video_key, label, output_dir):
    # --- CHECKPOINT & RACE PROTECTION ---
    out_file = os.path.join(output_dir, f"{pid}.h5")
    lock_file = os.path.join(output_dir, f"{pid}.lock")
    
    # 1. Checkpoint: Skip if output exists
    if os.path.exists(out_file):
        return
    
    # 2. Race Condition: Check lock
    if os.path.exists(lock_file):
        try:
            if time.time() - os.path.getmtime(lock_file) > 7200:
                print(f"⚠️ Removing stale lock for {pid}")
                try: os.remove(lock_file)
                except: pass
            else:
                print(f"🔒 Skipping {pid} (Locked by another instance)")
                return
        except OSError:
             pass
    
    # 3. Acquire Lock
    try:
        with open(lock_file, 'w') as f:
            f.write(str(time.time()))
    except OSError:
        print(f"🔒 Could not acquire lock for {pid}")
        return
        
    start_time = time.time()
    print(f"\n{'='*50}")
    print(f"📂 Processing {pid} (Key: {video_key}, Label: {label})")
    
    url = f"https://www.youtube.com/watch?v={video_key}"
    work_dir = os.path.join(CFG.TEMP_PATH, str(pid))
    if os.path.exists(work_dir):
        shutil.rmtree(work_dir)
    os.makedirs(work_dir, exist_ok=True)
    
    try:
        # 1. Download Video
        print(f"   ⬇️ Downloading {url}...")
        cmd = [
            'yt-dlp', 
            '-f', 'best[ext=mp4]/best',
            '-o', os.path.join(work_dir, 'video.%(ext)s'),
            '--write-subs', '--write-auto-subs', '--sub-lang', 'en', '--sub-format', 'vtt',
            '--no-check-certificate',
            '--ignore-errors',
            url
        ]
        subprocess.run(cmd, check=False)
        
        video_files = glob.glob(os.path.join(work_dir, 'video.*'))
        video_files = [f for f in video_files if not f.endswith('.vtt') and not f.endswith('.part') and not f.endswith('.ytdl')]
        
        video_path = None
        if video_files:
            video_path = video_files[0]
            print(f"   🎥 Found video: {os.path.basename(video_path)}")
        else:
            print(f"   ❌ Download failed for {pid} (no video file found)")
            return
        
        # Extract Audio
        audio_path = os.path.join(work_dir, 'audio.wav')
        try:
            subprocess.run([
                'ffmpeg', '-y', '-i', video_path, 
                '-vn', '-acodec', 'pcm_s16le', '-ar', '16000', '-ac', '1', 
                audio_path
            ], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        except Exception as e:
            print(f"   ⚠️ Audio extraction issue: {repr(e)}")
        
        # Find subtitles
        vtt_files = glob.glob(os.path.join(work_dir, '*.vtt'))
        transcript_text = ""
        
        if vtt_files:
            print(f"   📝 Found subtitles: {os.path.basename(vtt_files[0])}")
            try:
                with open(vtt_files[0], 'r', encoding='utf-8') as f:
                    lines = f.readlines()
                    seen_lines = set()
                    for line in lines:
                        line = line.strip()
                        if '-->' in line or line == 'WEBVTT' or not line: continue
                        if line not in seen_lines:
                            transcript_text += line + " "
                            seen_lines.add(line)
            except Exception as e:
                print(f"   ⚠️ Subtitle parse error: {repr(e)}")
        
        result = {'participant_id': pid, 'dataset': 'youtube_dvlog'}
        pipeline._safe_update(result, {'phq8_score': 10.0 if str(label).lower() == 'depression' else 0.0}, 'metadata')
        result['phq8_score'] = 10.0 if str(label).lower() == 'depression' else 0.0
        
        # 2. Process Audio (CORRECTED METHOD NAME)
        if os.path.exists(audio_path):
            print("   🎵 Processing Audio...")
            audio_feats = pipeline.audio.process_audio(audio_path, None)
            pipeline._safe_update(result, audio_feats, 'audio')
        
        # 3. Process Text (CORRECTED METHOD NAME)
        if transcript_text:
            print("   📜 Processing Text...")
            text_feats = pipeline.text.process_text(text=transcript_text)
            pipeline._safe_update(result, text_feats, 'text')
        else:
            print("   ⚠️ No text found")
            
        # 4. Process Video/Face
        if os.path.exists(video_path):
            print("   🎬 Processing Video/Face...")
            frames = pipeline.video.extractor.extract(video_path)
            
            video_feats = pipeline.video.process_frames(frames)
            pipeline._safe_update(result, video_feats, 'video')
            
            face_feats = pipeline.face.process_frames(frames)
            pipeline._safe_update(result, face_feats, 'face')

        # 5. Fusion & Tabular
        if 'audio_embedding' not in result: result['audio_embedding'] = np.zeros(768)
        if 'text_embedding' not in result: result['text_embedding'] = np.zeros(768)
        if 'video_embedding' not in result: result['video_embedding'] = np.zeros(768)
        if 'face_embedding' not in result: result['face_embedding'] = np.zeros(768)
        
        scalar_features = []
        EXPECTED_SCALARS = pipeline.cfg.EXPECTED_SCALAR_FEATURES if hasattr(pipeline.cfg, 'EXPECTED_SCALAR_FEATURES') else []
        from pipeline_fusion_main import EXPECTED_SCALAR_FEATURES
        
        for key in EXPECTED_SCALAR_FEATURES:
             val = result.get(key, 0.0)
             if isinstance(val, (np.ndarray, list)):
                 val = 0.0
             if isinstance(val, (int, float)) and not np.isnan(val):
                 scalar_features.append(pipeline.num_norm.transform(val, key))
             else:
                 scalar_features.append(0.0)
        
        scalar_features.append(0.5)
        
        scalar_tensor = torch.tensor(scalar_features, dtype=torch.float32).unsqueeze(0).to(pipeline.cfg.DEVICE)
        tabular_emb = pipeline.tabular_projector(scalar_tensor)
        result['tabular_embedding'] = tabular_emb.cpu().detach().numpy().flatten()
        
        embeddings_to_fuse = {
            'audio': result.get('audio_embedding'),
            'text': result.get('text_embedding'),
            'video': result.get('video_embedding'),
            'face': result.get('face_embedding'),
            'tabular': result['tabular_embedding']
        }
        
        quality = {
             'audio': float(result.get('audio_snr', 0.5) / 100),
             'text': min(1.0, len(transcript_text.split()) / 100),
             'video': result.get('video_quality_score', 0.5),
             'face': result.get('face_detection_rate', 0.5),
             'tabular': 1.0
        }
        
        if pipeline.fusion:
            try:
                result['fusion_embedding'] = pipeline.fusion.fuse(embeddings_to_fuse, quality)
            except Exception as e:
                print(f"Fusion error: {repr(e)}")
                result['fusion_embedding'] = np.zeros(768)
            
        # 6. Save
        out_file = os.path.join(output_dir, f"{pid}.h5")
        pipeline.save_to_h5([result], out_file)
        end_time = time.time()
        print(f"   ✅ Saved {pid}.h5 (⏱️ {end_time - start_time:.2f}s)")
        
    except Exception as e:
        print(f"   ❌ Error: {repr(e)}")
        import traceback
        traceback.print_exc()
    finally:
        if os.path.exists(work_dir):
            shutil.rmtree(work_dir)
        if os.path.exists(lock_file):
            try: os.remove(lock_file)
            except: pass


## Cell 10: Run Processing Loop

In [ ]:
from tqdm.notebook import tqdm
if not df_videos.empty:
    print(f"🚀 Starting processing of {len(df_videos)} videos...")
    
    for idx, row in tqdm(df_videos.iterrows(), total=df_videos.shape[0], desc='Processing Videos'):
        pid = f"yt_{row['video_id']}"
        key = row['key']
        label = row['label']
        
        output_file = os.path.join(CFG.OUTPUT_PATH, f"{pid}.h5")
        if os.path.exists(output_file):
            print(f"⏭️ Skipping {pid}, already exists.")
            continue
            
        process_youtube_participant(pipeline, pid, key, label, CFG.OUTPUT_PATH)
else:
    print("⚠️ No videos to process.")